# AgenticIQA Colab Demo
This notebook demonstrates how to run the AgenticIQA pipeline in Google Colab.
It covers environment setup, API configuration, and running a complete evaluation on a test image.

## 1. Setup Environment
First, we clone the repository and install all required dependencies including `IQA-PyTorch`.

In [ ]:
import os

# If running in Colab, clone the repo
if not os.path.exists('src/agentic/graph.py'):
    !git clone https://github.com/your-repo/easy_agenticIQA.git
    %cd easy_agenticIQA

# Install PyTorch with specific CUDA version
!pip install torch==2.3.0 torchvision==0.18.0 --extra-index-url https://download.pytorch.org/whl/cu121

# Install other requirements
!pip install -r requirements.txt

# Install IQA-PyTorch from source
!git clone https://github.com/chaofengc/IQA-PyTorch.git
!cd IQA-PyTorch && pip install -e .


## 2. Configure API Keys
Set up your API keys. We use Google Gemini by default in this repo, so we fetch `GOOGLE_API_KEY` from Colab Secrets.

In [ ]:
import os
import sys

try:
    from google.colab import userdata
    # Ensure you add 'GOOGLE_API_KEY' in the secrets tab (left sidebar)
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print("API Key loaded successfully from Colab secrets.")
except ImportError:
    print("Not running in Colab. Make sure your .env file is configured or set the environment variable manually.")
except userdata.SecretNotFoundError:
    print("Please add your GOOGLE_API_KEY to the Colab Secrets tab.")
    # Uncomment and replace if you want to hardcode (not recommended):
    # os.environ['GOOGLE_API_KEY'] = 'your-key-here'

# Initialize .env if it doesn't exist
!cp .env.example .env 2>/dev/null || echo "AGENTIC_ROOT=$(pwd)" > .env


## 3. Prepare a Test Image
Let's download an image to assess.

In [ ]:
import urllib.request
from PIL import Image
import matplotlib.pyplot as plt

os.makedirs('data/examples', exist_ok=True)
image_url = 'https://raw.githubusercontent.com/chaofengc/IQA-PyTorch/main/examples/k0464.jpg'
image_path = 'data/examples/test_image.jpg'
urllib.request.urlretrieve(image_url, image_path)

img = Image.open(image_path)
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.title('Test Image')
plt.show()


## 4. Run the AgenticIQA Pipeline
Here we execute the core pipeline which includes:
1. **Planner**: Decides the assessment plan based on the query
2. **Executor**: Runs VLM analysis and traditional IQA metrics
3. **Summarizer**: Combines VLM probabilities and IQA scores via fusion


In [ ]:
from src.agentic.graph import run_pipeline
import json

query = "What is the overall quality of this image? Are there any distortions?"
print(f"User Query: '{query}'\n")

try:
    result = run_pipeline(
        query=query,
        image_path=image_path,
        max_replan_iterations=2
    )
    
    summarizer_result = result.get('summarizer_result')
    if summarizer_result:
        output = {
            'Final Answer': summarizer_result.final_answer,
            'Quality Score': summarizer_result.quality_score,
            'Reasoning': summarizer_result.quality_reasoning
        }
        print("\n=== Pipeline Result ===")
        print(json.dumps(output, indent=2, ensure_ascii=False))
    else:
        print("Pipeline failed or returned no summarizer result.")
        if 'error' in result:
            print(f"Error: {result['error']}")
except Exception as e:
    print(f"Error executing pipeline: {e}")
